In [2]:
import geopandas as gpd

# Load your three prepared map layers
buildings = gpd.read_file('data/Islamabad_Buildings_Clean.gpkg')
density_grid = gpd.read_file('data/Islamabad_Density_Grid.gpkg')
fault_buffers = gpd.read_file('data/Margalla_Fault_Buffers.gpkg')

# Force all layers to use the exact same GPS coordinate system so they overlap perfectly
density_grid = density_grid.to_crs(buildings.crs)
fault_buffers = fault_buffers.to_crs(buildings.crs)

# Drop the buildings onto the grid to attach the neighborhood density count to each structure
bldg_with_density = gpd.sjoin(buildings, density_grid[['building_count', 'geometry']], how='left', predicate='intersects')

# Remove the temporary tracking column left over from the first join to prevent a crash
bldg_with_density = bldg_with_density.drop(columns=['index_right'])

# Drop those buildings onto the fault zones to attach the seismic risk weight
bldg_with_risk = gpd.sjoin(bldg_with_density, fault_buffers[['risk_level', 'risk_weight', 'geometry']], how='left', predicate='intersects')

# If a building is further than 10km from the fault, assign it a baseline risk weight of 0
bldg_with_risk['risk_weight'] = bldg_with_risk['risk_weight'].fillna(0)

# Scale the density count to a 0-to-3 range so it carries the same mathematical weight as the fault proximity
max_density = bldg_with_risk['building_count'].max()
bldg_with_risk['density_weight'] = (bldg_with_risk['building_count'] / max_density) * 3

# Add the two weights together to get the final structural vulnerability score
bldg_with_risk['vulnerability_score'] = bldg_with_risk['risk_weight'] + bldg_with_risk['density_weight']

# Strip out messy duplicate columns created during the spatial joins
columns_to_keep = ['geometry', 'building_count', 'risk_level', 'risk_weight', 'density_weight', 'vulnerability_score']
final_index = bldg_with_risk[columns_to_keep]

# Save the final master layer for Phase 1
final_index.to_file('data/Islamabad_Vulnerability_Index.gpkg', driver='GPKG')

# Print the highest score found to verify the math didn't break
max_score = round(final_index['vulnerability_score'].max(), 2)
print(f"Vulnerability index calculated. Maximum score found: {max_score}/6.0")

Vulnerability index calculated. Maximum score found: 4.54/6.0
